# A decorated dictionary cannot be empty

An intermediate class can define shared fields; a concrete dictionary must supply values.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
class MoneyBaseEnumEntity(BaseEnumEntity):
    """Define shared fields without declaring a concrete dictionary."""

    symbol: str = Field(description="Currency symbol")

print("Intermediate members:", MoneyBaseEnumEntity.members)
try:

    @entity(description="Missing vocabulary", domain=ShopDomain)
    class EmptyEnumEntity(MoneyBaseEnumEntity):
        """Intentionally declare no values."""
except ValueError as error:
    print(str(error))
else:
    raise AssertionError("An empty dictionary was registered")

Intermediate members: ()
EmptyEnumEntity declares no values; a closed dictionary states its values.


## What the result means

An undecorated intermediate can supply fields without any members. Registering a concrete dictionary with `@entity` requires a nonempty catalogue, so the empty subclass fails at registration. A state subclass follows a different catalogue-inheritance rule.